<a href="https://colab.research.google.com/github/sauravv19/A-Security-Analysis-of-Tf-Idf-And-Deep-Learning-Model-for-Hate-Speech-Detection-in-YouTube/blob/main/Hate%20Speech%20Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.6 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
from scipy.special import softmax
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline, AutoTokenizer, AutoModelForSequenceClassification, RobertaTokenizerFast, RobertaForSequenceClassification
from transformers.models.whisper.processing_whisper import WhisperProcessor
from transformers.models.whisper.modeling_whisper import WhisperForConditionalGeneration
from transformers.pipelines.automatic_speech_recognition import AutomaticSpeechRecognitionPipeline
from subprocess import run, PIPE
from typing import Self
from uuid import uuid4
from os import makedirs, remove

In [ ]:
MAX_SEQUENCE_LENGTH = 512
OVERLAP = 50
STRIDE = MAX_SEQUENCE_LENGTH - OVERLAP
MAX_NEW_TOKENS = 128
CHUNK_LENGTH = 30
VIDEO_URL = "https://youtu.be/U1S8bduxLKM?si=8EIVyKJHOeqZR5qu"

In [ ]:
class ASRHate:

    def __pipe__(self) -> Self:
      self.__asr_pipe__ = pipeline(
          "automatic-speech-recognition",
          model=self.__wh_model__,
          tokenizer=self.__wh_processor__.tokenizer,
          feature_extractor=self.__wh_processor__.feature_extractor,
          chunk_length_s=CHUNK_LENGTH,
          device=self.device,
          generate_kwargs={"max_new_tokens": MAX_NEW_TOKENS},
      )
      return self

    def __on_device__(self) -> Self:
      self.device = "cuda" if torch.cuda.is_available() else "cpu"
      self.__wh_model__.to(self.device)
      return self

    @staticmethod
    def __ytdlp__(video_url: str) -> str:
      makedirs("/content/outputs", exist_ok=True)
      out = f"/content/outputs/output-{str(uuid4())}.wav"
      proc = run(f'yt-dlp --extract-audio --audio-format wav --output {out} {video_url}', shell=True, stderr=PIPE, stdout=PIPE)
      if proc.returncode == 0:
        return out
      raise Exception(proc.stderr.decode(errors="ignore"))

    def transcribe(self, video_url: str) -> Self:
      self.__audio_path__ = ASRHate.__ytdlp__(video_url)
      self.transcription = self.__asr_pipe__(self.__audio_path__)
      return self

    def classify(self) -> Self:
      inputs = self.__classifier_tokenizer__(
          self.transcription["text"],
          max_length=MAX_SEQUENCE_LENGTH,
          padding="max_length",
          truncation=True,
          return_overflowing_tokens=True,
          stride=STRIDE,
          return_tensors="pt"
      )
      model_inputs = {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            # Add "token_type_ids": inputs["token_type_ids"].to(self.device) if your tokenizer returns them
      }
      with torch.no_grad():
        outputs = self.__classifier_model__(**model_inputs)
      if self.device == 'cuda':
        logits_np = outputs.logits.cpu().numpy()
      else:
        logits_np = outputs.logits.numpy()

      self.probabilities = softmax(logits_np, axis=1)
      return self

    def clean(self) -> None:
      remove(self.__audio_path__)
      print("removed", self.__audio_path__)
      delattr(self, "__audio_path__")

    def chunkwise(self):
      percentages = self.probabilities * 100
      percentages_list = percentages.tolist()
      print(percentages_list)
      label_0 = self.__classifier_model__.config.id2label[0].title()
      label_1 = self.__classifier_model__.config.id2label[1].title()
      for i, chunk_percentages in enumerate(percentages_list):
        print(f"Chunk {i}: {label_0} {chunk_percentages[0]:.2f}% {label_1} {chunk_percentages[1]:.2f}%")


    def overall(self) -> None:
      mean_probabilities_np = np.mean(self.probabilities, axis=0)
      mean_percentages_np = mean_probabilities_np * 100
      mean_percentages_list = mean_percentages_np.tolist()
      # Print the mean percentage for each label from the list
      for j, mean_percentage in enumerate(mean_percentages_list):
          label = self.__classifier_model__.config.id2label[j]
          print(f"Mean {label.title()}: {mean_percentage:.2f}%")



    def __init__(self):
      self.__wh_processor__ = WhisperProcessor.from_pretrained("openai/whisper-large-v3-turbo")
      self.__wh_model__ = WhisperForConditionalGeneration.from_pretrained(
          "openai/whisper-large-v3-turbo"
      )
      # Use AutoTokenizer for robustness
      self.__classifier_tokenizer__ = AutoTokenizer.from_pretrained("facebook/roberta-hate-speech-dynabench-r4-target")
      self.__classifier_model__ = RobertaForSequenceClassification.from_pretrained("facebook/roberta-hate-speech-dynabench-r4-target")
      self.__on_device__().__pipe__()

In [ ]:
asr_hate = ASRHate()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: facebook/roberta-hate-speech-dynabench-r4-target
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [ ]:
asr_hate.transcribe(VIDEO_URL).classify().clean()

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gener

removed /content/outputs/output-1f308f11-fa1d-49f0-93ec-d588bb5521b7.wav


In [ ]:
print(asr_hate.transcription['text'])

 Twitter clip the movie. They stitch together 5 movies worth of content just so retards can go post a 1 minute clip on twitter and create entirely different movie around it. Case in point, Mia Khalifa likes it because Paddoland is getting shit on. But this whole subplot starts with off screen event and then Superman can go help them because he needs to do another clip where he fights his clone, so***ing the t***as can post that clip and talk about how great the fights are and make up whole different movie around how this guy is gonna be bizarro. The ending has feels good clip. We get there by turning Jor-El and Lara into Ultramites but who cares, it's so sweet. Oh this 30 year old guy who was Superman for 3 years finds out doing good is good. Wow that's a good message. Oh by the way we see his parents for about 5 minutes total. This movie is what Martin Scrotum face and my mom see when they watch superhero movies. Stuff is constantly happening and more and more characters are introduce

In [ ]:
asr_hate.chunkwise()

[[6.934859752655029, 93.06513977050781], [22.75169563293457, 77.24830627441406], [92.6366958618164, 7.36329460144043], [94.35110473632812, 5.648890018463135], [98.86455535888672, 1.135447382926941], [98.00405883789062, 1.995951533317566], [95.40254211425781, 4.59745979309082], [99.37156677246094, 0.6284357905387878], [99.14694213867188, 0.8530558943748474], [98.77047729492188, 1.2295278310775757], [34.17627716064453, 65.82372283935547], [37.65388488769531, 62.34611129760742], [1.6616036891937256, 98.33838653564453], [14.752891540527344, 85.24710845947266], [6.2823333740234375, 93.71766662597656], [4.220675468444824, 95.77932739257812], [5.097420692443848, 94.90258026123047], [1.8735941648483276, 98.12641143798828], [7.9775919914245605, 92.02240753173828], [2.0394480228424072, 97.96055603027344], [2.884960651397705, 97.11503601074219], [7.474809169769287, 92.52519226074219], [2.423123598098755, 97.57687377929688], [5.638980388641357, 94.36102294921875], [6.97160530090332, 93.02838897705

In [ ]:
asr_hate.overall()

Mean Nothate: 36.46%
Mean Hate: 63.54%
